# 02 — Fine-tuning con QLoRA

Fine-tuning de Mistral-7B-Instruct-v0.3 usando QLoRA (cuantización 4-bit + LoRA adapters).

**Compute recomendado:** Google Colab A100 o Kaggle T4×2

In [ ]:
!pip install -q transformers peft bitsandbytes trl datasets accelerate wandb huggingface_hub

In [ ]:
import os
import torch
import yaml
from datasets import Dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Autenticación HuggingFace
# login(token=os.environ['HUGGINGFACE_TOKEN'])

# W&B opcional
# import wandb; wandb.login(key=os.environ['WANDB_API_KEY'])

## 1. Configuración

In [ ]:
CFG = {
    'model_name': 'mistralai/Mistral-7B-Instruct-v0.3',
    'dataset_path': '../data/dataset.jsonl',
    'output_dir': '../outputs',
    'hub_model_id': 'lopezinsua/mistral-7b-code-reviewer-es',
    # QLoRA
    'load_in_4bit': True,
    'bnb_4bit_quant_type': 'nf4',
    'bnb_4bit_compute_dtype': 'float16',
    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'v_proj'],
    # Training
    'num_train_epochs': 3,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 4,
    'learning_rate': 2e-4,
    'warmup_ratio': 0.03,
    'lr_scheduler_type': 'cosine',
    'max_seq_length': 512,
}

## 2. Dataset

In [ ]:
import json

PROMPT = "<s>[INST] {instruction}\n\n{input} [/INST] {output}</s>"

def load_and_format(path: str) -> Dataset:
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                text = PROMPT.format(**r) if r.get('input') else \
                       f"<s>[INST] {r['instruction']} [/INST] {r['output']}</s>"
                records.append({'text': text})
    return Dataset.from_list(records)

dataset = load_and_format(CFG['dataset_path'])
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])} | Test: {len(split['test'])}")

## 3. Modelo con cuantización 4-bit

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CFG['load_in_4bit'],
    bnb_4bit_quant_type=CFG['bnb_4bit_quant_type'],
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'], trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    CFG['model_name'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
print(f"Modelo cargado | dtype: {model.dtype}")

## 4. LoRA adapters

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    target_modules=CFG['target_modules'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir=CFG['output_dir'],
    num_train_epochs=CFG['num_train_epochs'],
    per_device_train_batch_size=CFG['per_device_train_batch_size'],
    gradient_accumulation_steps=CFG['gradient_accumulation_steps'],
    learning_rate=CFG['learning_rate'],
    warmup_ratio=CFG['warmup_ratio'],
    lr_scheduler_type=CFG['lr_scheduler_type'],
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id=CFG['hub_model_id'],
    report_to='wandb',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=CFG['max_seq_length'],
    packing=False,
)

trainer.train()

In [ ]:
# Publicar en HuggingFace Hub
trainer.push_to_hub()
tokenizer.push_to_hub(CFG['hub_model_id'])
print(f"Modelo publicado: https://huggingface.co/{CFG['hub_model_id']}")